# 03 - Backpropagation & Computational Graphs
Let's build a tiny autograd engine in Python to demonstrate backprop!

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._prev = set(_children)
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

# Example Graph
x = Value(2.0)
y = Value(-3.0)
z = Value(10.0)

# f = x*y + z
q = x * y
f = q + z

f.backward()
print(f'f = {f.data}')
print(f'Gradient wrt x: {x.grad}')
print(f'Gradient wrt y: {y.grad}')
print(f'Gradient wrt z: {z.grad}')